In [ ]:
import os
import getpass

os.environ["NVIDIA_API_KEY"]= getpass.getpass("Enter your NVIDIA API key")

Enter your NVIDIA API key··········


In [ ]:
!pip install -qU openai



In [ ]:
!pip install --upgrade openai

In [ ]:
from openai import OpenAI
client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
    api_key= os.environ["NVIDIA_API_KEY"]
)

In [ ]:
math_prompt = """ What is 12.99+13.1 equal to ?"""

completion = client.chat.completions.create(
model="qwen/qwen3-next-80b-a3b-thinking",
messages=[{"role":"user","content":math_prompt}],
temperature=0.6,
top_p=0.7,
max_tokens=4096,
stream=True
)

for chunk in completion:
  reasoning = getattr(chunk.choices[0].delta, "reasoning_content", None)
  if reasoning:
    print(reasoning, end="")
  if chunk.choices[0].delta.content is not None:
    print(chunk.choices[0].delta.content, end="")

Okay, the user is asking what 12.99 plus 13.1 equals. Let me start by recalling how to add decimals. First, I need to align the decimal points. So, 12.99 and 13.1. Wait, 13.1 is the same as 13.10, right? That way, both numbers have two decimal places. So writing them out:

  12.99
+ 13.10
--------

Now, adding the hundredths place: 9 + 0 = 9. Then the tenths place: 9 + 1 = 10. So write down 0 and carry over 1. Next, the ones place: 2 + 3 = 5, plus the carried over 1 makes 6. Then the tens place: 1 + 1 = 2. So putting it all together, it should be 26.09.

Wait, let me check again. Maybe I should do it step by step. 12.99 + 13.1. Let's convert them to fractions to verify. 12.99 is 1299/100, and 13.1 is 131/10, which is 1310/100. Adding them: 1299 + 1310 = 2609. So 2609/100 = 26.09. Yeah, that matches.

Alternatively, think of it as 12.99 + 13 = 25.99, then add 0.1 more: 25.99 + 0.1 = 26.09. That works too. So the answer is 26.09. The user might be a student doing homework, or maybe someo

In [ ]:
!pip install -qU "openai-agents[litellm]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.4/81.4 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.4/144.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.3/272.3 kB 14.3 MB/s eta 0:00:00


In [ ]:
from openai import AsyncOpenAI

client = AsyncOpenAI(
    base_url = "https://integrate.api.nvidia.com/v1",
    api_key= os.environ["NVIDIA_API_KEY"]
)


In [ ]:
from agents import function_tool
import os

@function_tool
async def display_file(filename:str) -> str:
  if not os.path.exists(filename):
    return f"File {filename} does not exist"
  with open(filename, "r") as f:
    return f.read()


@function_tool
def write_file(filename:str, content: str) -> str:
  with open(filename, "a", encoding="utf-8") as f:
    f.write(content + '\n')
  return f"Wrote to {filename}"




In [ ]:
from agents import Agent, OpenAIChatCompletionsModel, ModelSettings

agent = Agent(
    name="Notes Assistant",
    instructions="You are a helpful assistant. You take notes and save them to notes.txt. you can also read from notes.txt",
    model=OpenAIChatCompletionsModel(model="qwen/qwen3-next-80b-a3b-thinking", openai_client=client),
    tools=[display_file, write_file]
)

In [ ]:
from agents import Runner

result= await Runner.run(agent, "I am excited to use Qwen3 with NVIDIA NIM, can you take a note of this ?")

In [ ]:
result= await Runner.run(agent, "Can you read me what I have in my notes ? ")
print(result.final_output)

Your notes contain the following:

I am excited to use Qwen3 with NVIDIA NIM  
I am excited to use Qwen3 with NVIDIA NIM  
I am excited to use Qwen3 with NVIDIA NIM
